# Học phân loại nhiều lớp với dữ liệu hoa Iris

Notebook này dạy từng bước: KNN, logistic regression nhiều lớp, SVM và PCA. Mỗi bước có phần giải thích, việc cần làm và kết quả mong đợi để bạn tự kiểm tra.

Dữ liệu: 150 bông hoa thuộc 3 loài (setosa, versicolor, virginica). Mỗi bông có 4 số đo tính bằng cm: chiều dài và chiều rộng của đài hoa (sepal), chiều dài và chiều rộng của cánh hoa (petal). Nguồn: bộ Iris của Fisher (1936), lấy từ scikit-learn.

Chuẩn bị:
1. Đặt file data2.csv cùng thư mục với notebook.
2. Cài thư viện nếu chưa có: pip install pandas numpy matplotlib seaborn scikit-learn

Cách làm: đọc phần giải thích, viết code vào ô trống, chạy bằng Shift + Enter, rồi so sánh với kết quả mong đợi. Nếu kết quả khác, đọc lại đề bài và thử lại trước khi xem đáp án.

## Bước 0: Import thư viện

Mục tiêu: nạp các thư viện dùng trong toàn bộ notebook. Ô này đã viết sẵn, bạn chỉ cần chạy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

## Bước 1: Đọc dữ liệu

Mục tiêu: đưa file CSV vào DataFrame và kiểm tra kích thước.

Kiến thức: pd.read_csv đọc file thành bảng. df.shape trả về (số dòng, số cột).

Kết quả mong đợi: (150, 5).

In [ ]:
# Viết code ở đây:
# 1. Đọc file data2.csv vào biến df
# 2. In df.shape


## Bước 2: Xem tổng quan

Mục tiêu: xem vài dòng đầu, kiểu dữ liệu, giá trị thiếu và số mẫu mỗi loài.

Kiến thức: value_counts() đếm số lần xuất hiện của mỗi giá trị. Một bộ dữ liệu cân bằng có số mẫu mỗi lớp gần bằng nhau.

Kết quả mong đợi: 4 cột số, 1 cột chữ (species), 0 ô trống, mỗi loài 50 mẫu.

Câu hỏi: dữ liệu cân bằng thì accuracy có đáng tin hơn so với dữ liệu lệch không? Vì sao?

In [ ]:
# Viết code ở đây:
# 1. In 5 dòng đầu
# 2. Gọi df.info()
# 3. In tổng số ô trống
# 4. In số mẫu của từng loài trong cột species


## Bước 3: So sánh trung bình theo loài

Mục tiêu: tìm số đo nào tách các loài rõ nhất.

Kiến thức: groupby("species").mean() tính trung bình của từng cột cho mỗi loài. Cột nào có trung bình rất khác nhau giữa các loài thì có khả năng phân loại tốt.

Kết quả mong đợi: petal_length và petal_width khác nhau nhiều nhất: setosa có petal_length trung bình 1,46, versicolor 4,26, virginica 5,55. Sepal_width gần như giống nhau giữa các loài (2,77 đến 3,43).

Câu hỏi: nhìn bảng, bạn đoán cột nào sẽ quan trọng nhất cho mô hình?

In [ ]:
# Viết code ở đây:
# 1. Nhóm df theo species và tính trung bình
# 2. Làm tròn 2 chữ số rồi in ra


## Bước 4: Pairplot

Mục tiêu: nhìn mọi cặp số đo trên cùng một hình.

Kiến thức: sns.pairplot vẽ scatter cho mỗi cặp cột và histogram trên đường chéo. hue tô màu theo nhãn.

Kết quả mong đợi: setosa tách hẳn khỏi hai loài còn lại ở các hình có petal. Versicolor và virginica chồng lên nhau một phần.

Câu hỏi: nếu chỉ được dùng một hình scatter để phân loại, bạn chọn cặp cột nào?

In [ ]:
# Viết code ở đây:
# 1. Gọi sns.pairplot(df, hue='species')
# 2. plt.show()


## Bước 5: Tách X, y và chia train/test

Mục tiêu: tách feature và nhãn, rồi giữ 30% làm tập test.

Kiến thức: stratify=y giữ tỷ lệ các loài giống nhau ở hai tập. Với 150 mẫu, test 30% cho 45 mẫu, mỗi loài 15.

Kết quả mong đợi: X_train có 105 dòng, X_test có 45 dòng, mỗi loài trong y_test có 15 mẫu.

In [ ]:
# Viết code ở đây:
# 1. X là 4 cột số (bỏ species), y là cột species
# 2. Chia train/test với test_size=0.3, random_state=42, stratify=y
# 3. In kích thước X_train, X_test và value_counts của y_test


## Bước 6: Baseline

Mục tiêu: biết mức accuracy thấp nhất mà một mô hình phải vượt qua.

Kiến thức: DummyClassifier(strategy="most_frequent") luôn đoán lớp xuất hiện nhiều nhất. Với 3 lớp cân bằng, nó đạt khoảng 1/3.

Kết quả mong đợi: accuracy khoảng 0,333.

In [ ]:
# Viết code ở đây:
# 1. Tạo DummyClassifier(strategy='most_frequent')
# 2. Fit trên tập train, in accuracy trên tập test (làm tròn 3 chữ số)


## Bước 7: Chuẩn hóa dữ liệu

Mục tiêu: đưa các cột về cùng thang đo trước khi dùng mô hình dựa trên khoảng cách.

Kiến thức: StandardScaler trừ trung bình rồi chia độ lệch chuẩn, nên mỗi cột có trung bình 0 và độ lệch chuẩn 1. Chỉ fit trên tập train, sau đó dùng transform cho tập test. Nếu fit trên toàn bộ dữ liệu, thông tin của tập test sẽ rò rỉ vào mô hình (data leakage).

Kết quả mong đợi: trung bình của X_train_s bằng 0, độ lệch chuẩn bằng 1 ở cả 4 cột. Trên X_test_s, các con số chỉ gần 0 và gần 1.

Câu hỏi: tại sao trên tập test trung bình không đúng bằng 0?

In [ ]:
# Viết code ở đây:
# 1. Tạo StandardScaler, fit_transform trên X_train để được X_train_s
# 2. transform X_test để được X_test_s
# 3. In trung bình và độ lệch chuẩn từng cột của cả hai (làm tròn 2 chữ số)


## Bước 8: KNN với k = 5

Mục tiêu: huấn luyện mô hình K-Nearest Neighbors đầu tiên.

Kiến thức: KNN không học tham số. Khi dự đoán một điểm mới, nó tìm k điểm gần nhất trong tập train và lấy nhãn chiếm đa số. Khoảng cách mặc định là Euclid, nên thang đo của các cột ảnh hưởng trực tiếp đến kết quả.

Kết quả mong đợi: accuracy khoảng 0,911, tức là đúng 41 trên 45 mẫu.

In [ ]:
# Viết code ở đây:
# 1. Tạo KNeighborsClassifier(n_neighbors=5)
# 2. Fit trên X_train_s, y_train
# 3. In accuracy trên X_test_s


## Bước 9: Chọn k bằng cross-validation

Mục tiêu: thử nhiều giá trị k và chọn k tốt nhất mà không nhìn vào tập test.

Kiến thức: make_pipeline(StandardScaler(), KNeighborsClassifier(k)) gộp chuẩn hóa và mô hình, nên trong mỗi fold scaler chỉ fit trên phần train của fold đó. cross_val_score chạy 5 fold trên tập train. k nhỏ dễ overfit, k quá lớn thì mô hình quá đơn giản.

Kết quả mong đợi: k = 7 cho CV accuracy cao nhất, khoảng 0,962. Khi k lớn hơn 15, accuracy giảm dần xuống khoảng 0,876 ở k = 25.

Câu hỏi: vì sao không chọn k dựa trên accuracy của tập test?

In [ ]:
# Viết code ở đây:
# 1. Tạo cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# 2. Lặp k trong range(1, 26, 2): tính trung bình cross_val_score của pipeline trên X_train, y_train
# 3. Lưu kết quả vào dict, in ra, vẽ đường k với accuracy
# 4. Tìm best_k có accuracy cao nhất (nếu bằng nhau thì lấy k nhỏ nhất)


## Bước 10: Confusion matrix

Mục tiêu: xem mô hình nhầm loài nào với loài nào.

Kiến thức: confusion matrix có hàng là nhãn thật, cột là nhãn dự đoán. Các ô ngoài đường chéo là lỗi. classification_report cho precision, recall và F1 của từng lớp.

Kết quả mong đợi: setosa đúng hết. Accuracy khoảng 0,93. Virginica có recall 0,80, tức là 3 bông virginica bị đoán thành versicolor.

Câu hỏi: lỗi (nếu có) rơi vào cặp loài nào? Có khớp với pairplot ở bước 4 không?

In [ ]:
# Viết code ở đây:
# 1. Tạo pipeline StandardScaler + KNN với best_k, fit trên X_train
# 2. Dự đoán X_test, lưu vào pred_knn
# 3. Vẽ ConfusionMatrixDisplay.from_predictions(y_test, pred_knn)
# 4. In classification_report


## Bước 11: Logistic regression nhiều lớp

Mục tiêu: dùng logistic regression cho 3 lớp và đọc xác suất dự đoán.

Kiến thức: với nhiều lớp, sklearn dùng hàm softmax: mỗi lớp có một bộ hệ số, và xác suất của 3 lớp cộng lại bằng 1. coef_ có kích thước (số lớp, số feature).

Kết quả mong đợi: accuracy khoảng 0,911. coef_ có kích thước (3, 4). Với bông hoa đầu tiên, xác suất virginica khoảng 0,899 và nhãn thật là virginica.

Câu hỏi: với bông hoa đầu tiên trong tập test, mô hình chắc chắn đến mức nào?

In [ ]:
# Viết code ở đây:
# 1. Tạo pipeline StandardScaler + LogisticRegression(max_iter=1000), fit trên X_train
# 2. In accuracy trên X_test
# 3. In kích thước coef_ của bước logisticregression (gợi ý: logreg[-1].coef_.shape)
# 4. In predict_proba của dòng đầu tiên trong X_test (làm tròn 3 chữ số) cùng logreg.classes_


## Bước 12: SVM tuyến tính và SVM RBF

Mục tiêu: so sánh SVM với kernel tuyến tính và kernel RBF.

Kiến thức: SVM tìm đường biên có lề (margin) rộng nhất giữa các lớp. Kernel linear cho đường biên thẳng. Kernel rbf cho đường biên cong. C lớn thì phạt lỗi mạnh, lề hẹp hơn và dễ overfit hơn.

Kết quả mong đợi: linear: CV khoảng 0,962, test khoảng 0,911. rbf: CV khoảng 0,971, test khoảng 0,933.

In [ ]:
# Viết code ở đây:
# 1. Với kernel trong ['linear', 'rbf']: tạo pipeline StandardScaler + SVC(kernel=kernel)
# 2. Tính CV accuracy trên X_train (dùng cv ở bước 9) và test accuracy
# 3. In cả hai con số (làm tròn 3 chữ số)


## Bước 13: Tìm C và gamma bằng GridSearchCV

Mục tiêu: tự động thử nhiều tổ hợp tham số và chọn tổ hợp tốt nhất theo cross-validation.

Kiến thức: GridSearchCV thử mọi tổ hợp trong param_grid. Tên tham số trong pipeline có dạng tên_bước__tên_tham_số, ví dụ svc__C. best_params_ và best_score_ cho tổ hợp tốt nhất.

Kết quả mong đợi: best params là C = 10 và gamma = 0,01, CV accuracy khoảng 0,981, test accuracy khoảng 0,911. Test accuracy thấp hơn SVM RBF mặc định ở bước 12 dù CV cao hơn.

Câu hỏi: CV accuracy tăng nhưng test accuracy lại giảm. Với tập test chỉ 45 mẫu, chênh lệch 1 mẫu có ý nghĩa không? Bạn tin con số nào hơn?

In [ ]:
# Viết code ở đây:
# 1. Tạo pipeline StandardScaler + SVC(kernel='rbf')
# 2. param_grid = {'svc__C': [0.1, 1, 10, 100], 'svc__gamma': ['scale', 0.01, 0.1, 1]}
# 3. GridSearchCV(pipeline, param_grid, cv=cv), fit trên X_train
# 4. In best_params_, best_score_ và accuracy của best_estimator_ trên X_test


## Bước 14: PCA: giảm từ 4 chiều xuống 2 chiều

Mục tiêu: tìm 2 trục mới giữ được nhiều thông tin nhất của 4 cột ban đầu.

Kiến thức: PCA (principal component analysis) tạo các trục mới là tổ hợp tuyến tính của các cột cũ. Trục thứ nhất giữ nhiều phương sai nhất. explained_variance_ratio_ cho biết mỗi trục giữ bao nhiêu phần trăm phương sai. PCA cần chuẩn hóa trước, nếu không cột có đơn vị lớn sẽ chiếm ưu thế.

Kết quả mong đợi: PC1 giữ khoảng 73,0% phương sai, PC2 khoảng 22,9%, cộng lại khoảng 95,8%. PC1 có trọng số lớn ở petal_length, petal_width và sepal_length. PC2 chủ yếu là sepal_width (0,923).

Câu hỏi: 2 trục đầu giữ bao nhiêu phần trăm thông tin? Có đủ để vẽ dữ liệu lên mặt phẳng không?

In [ ]:
# Viết code ở đây:
# 1. Chuẩn hóa toàn bộ X bằng StandardScaler (bước này chỉ để khám phá, không đánh giá mô hình)
# 2. Fit PCA() trên dữ liệu đã chuẩn hóa
# 3. In explained_variance_ratio_ và tổng tích lũy (np.cumsum), làm tròn 3 chữ số
# 4. In pca.components_ của 2 trục đầu dưới dạng DataFrame với columns=X.columns


## Bước 15: Vẽ dữ liệu trên 2 trục PCA

Mục tiêu: nhìn cả 150 bông hoa trên một hình 2 chiều.

Kiến thức: X_pca[:, 0] là tọa độ trên PC1, X_pca[:, 1] là tọa độ trên PC2. sns.scatterplot với hue=y tô màu theo loài.

Kết quả mong đợi: 3 cụm theo PC1. Setosa nằm riêng một phía. Versicolor và virginica nằm gần nhau và chạm nhau ở giữa.

In [ ]:
# Viết code ở đây:
# 1. Vẽ scatter với x = X_pca[:, 0], y = X_pca[:, 1], hue = y
# 2. Đặt nhãn trục PC1 và PC2 kèm phần trăm phương sai


## Bước 16: Mô hình trên 2 trục PCA

Mục tiêu: kiểm tra xem bỏ bớt 2 chiều có làm giảm accuracy không.

Kiến thức: đặt PCA(n_components=2) vào pipeline giữa scaler và mô hình. Khi đó PCA chỉ fit trên tập train trong mỗi fold.

Kết quả mong đợi: 2 PC: CV accuracy khoảng 0,895, test accuracy khoảng 0,889. Logistic với 4 cột có CV khoảng 0,981.

Câu hỏi: giảm chiều có lợi gì khi dữ liệu có hàng trăm cột thay vì 4 cột?

In [ ]:
# Viết code ở đây:
# 1. Tạo pipeline StandardScaler + PCA(n_components=2) + LogisticRegression(max_iter=1000)
# 2. Tính CV accuracy trên X_train và test accuracy
# 3. So sánh với logistic dùng đủ 4 cột ở bước 11


## Bước 17: Bảng so sánh các mô hình

Mục tiêu: ghép kết quả của tất cả mô hình vào một bảng.

Kiến thức: với dữ liệu nhỏ như Iris, test accuracy trên 45 mẫu dao động mạnh: sai thêm 1 mẫu là mất 2,2 điểm phần trăm. Vì vậy nên xem cả CV accuracy và độ lệch chuẩn của nó.

Kết quả mong đợi: các mô hình (trừ baseline) có CV mean từ 0,895 đến 0,981. KNN k = 7 có test accuracy cao nhất (0,933), còn Logistic và SVM RBF tuned có CV mean cao nhất (0,981).

Câu hỏi: các mô hình có khác nhau thật sự không, hay chênh lệch nằm trong độ lệch chuẩn?

In [ ]:
# Viết code ở đây:
# 1. Tạo dict models gồm: Baseline, KNN (best_k), Logistic, SVM linear, SVM RBF (tham số tốt nhất), Logistic trên 2 PC
# 2. Với mỗi mô hình: tính cross_val_score trên X_train (cv=cv), lấy mean và std; fit rồi tính test accuracy
# 3. Lưu vào list rows, tạo DataFrame và in ra


## Bước 18: Tự thử nghiệm

Không có code mẫu ở bước này. Hãy tự chọn ít nhất hai việc sau:

1. Chạy KNN với best_k nhưng không chuẩn hóa dữ liệu. Accuracy thay đổi thế nào? Với Iris, các cột đều tính bằng cm, vậy chuẩn hóa có còn quan trọng không?
2. Chỉ dùng 2 cột petal_length và petal_width. So sánh accuracy với khi dùng đủ 4 cột.
3. Đổi KNeighborsClassifier(weights='distance'). Kết quả có khác không?
4. Viết ba câu tiếng Anh tóm tắt bảng ở bước 17. Ví dụ mở đầu: "All models except the baseline reached a cross-validated accuracy above ..."
